# 第8段階：未使用テストseedによる最終評価

第7段階で固定した9候補を、候補選択に使っていないseed `50001`--`50005`で評価する。候補の再選択は行わず、事前指定した主候補と副候補を区別して可視化する。

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.rcParams['font.family'] = ['Hiragino Sans', 'Yu Gothic', 'Noto Sans CJK JP', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

In [ ]:
def find_repo_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / 'experiment_protocols').is_dir() and (path / 'analysis').is_dir():
            return path
    raise FileNotFoundError('リポジトリのルートを特定できません。')

repo_root = find_repo_root(Path.cwd())
stage_root = repo_root / 'experiments/summer_2026/stage8_final_evaluation'

# 特定の分析を使う場合はPathを代入する。Noneなら完了済みの最新分析を探す。
ANALYSIS_ROOT = None
if ANALYSIS_ROOT is None:
    candidates = []
    for path in stage_root.glob('*/final_evaluation_analysis_v01'):
        manifest_path = path / 'analysis_manifest.json'
        if manifest_path.is_file():
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            if manifest.get('status') == 'completed':
                candidates.append(path)
    if not candidates:
        raise FileNotFoundError('完了済みの第8段階分析がありません。実験・正式分析後に再実行してください。')
    ANALYSIS_ROOT = sorted(candidates)[-1]
else:
    ANALYSIS_ROOT = Path(ANALYSIS_ROOT)

tables = ANALYSIS_ROOT / 'tables'
figures = ANALYSIS_ROOT / 'figures'
figures.mkdir(exist_ok=True)
final_results = pd.read_csv(tables / 'final_candidate_results.csv')
seed_results = pd.read_csv(tables / 'candidate_seed_performance.csv')
validation_test = pd.read_csv(tables / 'validation_test_comparison.csv')
network_conclusions = pd.read_csv(tables / 'network_conclusions.csv')
decision = json.loads((ANALYSIS_ROOT / 'decision.json').read_text(encoding='utf-8'))
print('analysis:', ANALYSIS_ROOT.relative_to(repo_root))
print('status:', decision['status'], '| candidates_reselected:', decision['candidates_reselected'])

## 1. ネットワーク別の主結論

`selection_order=1`は最終テスト前に固定した主候補である。Facebookは第7段階で適格候補がなかったため、探索的結果として読む。

In [ ]:
display(network_conclusions[[
    'network', 'evidence_scope', 'primary_condition_id',
    'primary_certainty', 'primary_effectiveness',
    'primary_test_mean_jcum', 'none_relative_suppression',
    'none_relative_ci_low', 'none_relative_ci_high',
    'none_interpretation', 'conclusion_code'
]])

## 2. 無介入に対する相対抑制率

点は相対抑制率 $\eta$、横線は95% CIである。丸は主候補、四角は別領域の副候補を表す。0より右が無介入より利己的行動を減らした方向である。

In [ ]:
plot_data = final_results.sort_values(['network', 'selection_order'], ascending=[True, False]).reset_index(drop=True)
colors = {'ba1000': '#2563eb', 'facebook': '#d97706', 'wiki_vote': '#059669'}
fig, ax = plt.subplots(figsize=(10, 6))
for y, row in plot_data.iterrows():
    estimate = row['none_relative_suppression']
    low = row['none_relative_ci_low']
    high = row['none_relative_ci_high']
    marker = 'o' if row['selection_order'] == 1 else 's'
    ax.errorbar(
        estimate, y, xerr=[[estimate - low], [high - estimate]],
        fmt=marker, color=colors[row['network']], capsize=3, markersize=6
    )
ax.axvline(0, color='#111827', linewidth=1, linestyle='--')
ax.set_yticks(
    range(len(plot_data)),
    [f"{r.network}  #{int(r.selection_order)}  {r.condition_id}" for r in plot_data.itertuples()]
)
ax.set_xlabel('無介入に対する相対抑制率 η（95% CI）')
ax.set_title('未使用テストseedにおける最終候補の抑制効果')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
fig.savefig(figures / '01_候補別最終相対抑制率.png', bbox_inches='tight')
plt.show()

## 3. 春学期代表条件に対する追加改善

`legacy_balance`との差は、介入そのものの効果ではなく、最適化した候補による追加改善を表す。

In [ ]:
display(final_results[[
    'network', 'condition_id', 'selection_order', 'reporting_role',
    'legacy_balance_relative_suppression',
    'legacy_balance_relative_ci_low', 'legacy_balance_relative_ci_high',
    'legacy_balance_interpretation'
]].sort_values(['network', 'selection_order']))

## 4. 検証値と最終テスト値

第7段階の検証値と第8段階のテスト値を比較し、候補選択による楽観性を確認する。この差を使って候補を選び直してはならない。

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for network, group in validation_test.groupby('network', sort=True):
    ax.scatter(
        group['stage7_validation_mean_jcum'], group['stage8_test_mean_jcum'],
        label=network, color=colors[network], s=55, alpha=0.85
    )
limits = [
    min(validation_test['stage7_validation_mean_jcum'].min(), validation_test['stage8_test_mean_jcum'].min()),
    max(validation_test['stage7_validation_mean_jcum'].max(), validation_test['stage8_test_mean_jcum'].max())
]
ax.plot(limits, limits, color='#111827', linestyle='--', linewidth=1)
ax.set_xlim(limits); ax.set_ylim(limits)
ax.set_xlabel('第7段階：検証seedの平均 Jcum')
ax.set_ylabel('第8段階：未使用テストseedの平均 Jcum')
ax.set_title('候補選択時と最終評価時の比較')
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
fig.savefig(figures / '02_検証値と最終テスト値の比較.png', bbox_inches='tight')
plt.show()
display(validation_test.sort_values(['network', 'selection_order']))

## 5. 主候補のseedブロック別効果

平均だけでなく、5つの未使用seedで効果方向が揃っているかを確認する。

In [ ]:
primary_seed = seed_results[seed_results['selection_order'] == 1].copy()
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, network in zip(axes, ['ba1000', 'facebook', 'wiki_vote']):
    group = primary_seed[primary_seed['network'] == network].sort_values('simulator_seed')
    ax.plot(
        group['simulator_seed'].astype(str),
        group['relative_suppression_vs_none'],
        marker='o', color=colors[network]
    )
    ax.axhline(0, color='#111827', linewidth=1, linestyle='--')
    ax.set_title(network)
    ax.set_xlabel('最終テストseed')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(alpha=0.2)
axes[0].set_ylabel('無介入に対する相対抑制率 η')
fig.suptitle('事前指定した主候補のseedブロック別効果')
plt.tight_layout()
fig.savefig(figures / '03_主候補のseed別効果.png', bbox_inches='tight')
plt.show()

## 6. 設計変数空間での候補位置

候補点は評価済みの点だけを示す。候補間の未評価領域まで良好だとは解釈しない。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), sharex=True, sharey=True)
for ax, network in zip(axes, ['ba1000', 'facebook', 'wiki_vote']):
    group = final_results[final_results['network'] == network]
    for row in group.itertuples():
        marker = 'o' if row.selection_order == 1 else 's'
        ax.scatter(
            row.certainty, row.effectiveness,
            s=90, marker=marker, color=colors[network], edgecolor='white', linewidth=0.8
        )
        ax.annotate(f'#{int(row.selection_order)}', (row.certainty, row.effectiveness), xytext=(5, 4), textcoords='offset points')
    ax.set_title(network)
    ax.set_xlabel('確実性')
    ax.grid(alpha=0.2)
axes[0].set_ylabel('有効性')
axes[0].set_xlim(0.48, 1.02); axes[0].set_ylim(0.48, 1.02)
fig.suptitle('第7段階で固定した候補の設計変数')
plt.tight_layout()
fig.savefig(figures / '04_最終候補の設計変数位置.png', bbox_inches='tight')
plt.show()